Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0);
you may not use this file except in compliance with the Apache 2.0 license.
You may obtain a copy of the Apache 2.0 license at:
https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0
International License (CC-BY). You may obtain a copy of the CC-BY license at:
https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and
materials distributed here under the Apache 2.0 or CC-BY licenses are
distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND,
either express or implied. See the licenses for the specific language governing
permissions and limitations under those licenses.

This is not an official Google product.

In [ ]:
# @title Imports (run this first)

import matplotlib.pyplot as plt
import numpy as np
import sympy

# An uncertainty principle inequality: a Hermite polynomials approach

Given a function $f: \mathbb{R}\rightarrow \mathbb{R}$, define the Fourier
transform $\widehat f(\xi) := \int_\mathbb{R} f(x) e^{-2\pi i x\xi}\ dx$ and $$
A(f) := \inf \{r > 0: f(x) \geq 0 \hbox{ for all } |x| \geq r \}.$$ Let $C_4$ be
the largest constant satisfying $$ A(f) A(\widehat f) \geq C_4$$ for all even
$f$ with $\max ( f(0), \widehat f(0)) < 0$. It is known that $$ 0.2025 \leq C_4
\leq 0.3523;$$ see Theorem 2 of
[Gonçalves et al. (2017)](https://www.sciencedirect.com/science/article/pii/S0022247X17301804)
(the upper bound is stated as $0.353$ in the paper, but rounding their solution
to the fourth digit gives $0.3523$). We improved the upper bound to $C_4 \leq
0.3521$ with a similar linear combination as in
[Gonçalves et al. (2017)](https://www.sciencedirect.com/science/article/pii/S0022247X17301804)
but with refined constants that were found by AlphaEvolve.

We address the problem using two different formulations described in the paper. These two formulations lead to known upper bounds $C_4 \leq 0.3523$ and
$C_4 \leq 0.3284$, respectively. Using AlphaEvolve, we improve these bounds to
$C_4 \leq 0.3521$ and $C_4 \leq 0.3216$, respectively.

First, following the method in
[Gonçalves et al. (2017)](https://www.sciencedirect.com/science/article/pii/S0022247X17301804),
we consider test functions of the form $f(x) = P(x)e^{-\pi x^2}$, where $P(x)$
is an even polynomial. For this form, the Fourier transform $\hat f(\xi)$ has
the structure $P(\xi)e^{-\pi \xi^2}$, which implies $A(f) = A(\hat f)$ is the
largest positive root of $P(x)$ (for functions that are positive for large
$|x|$). The inequality then becomes $C_4 \le (A(f))^2$.

The polynomial $P(x)$ is chosen as a linear combination of Hermite polynomials
of even order $H_{4k}(x)$, specifically $P(x) = \sum_{k=0}^m c_k H_{4k}(x)$. The
goal is to find coefficients $c_k$ that minimize the largest positive root of
$P(x)$ while satisfying certain conditions on $P(x)$ (related to the
$f(0)<0$ condition and $P(x)>0$ for large $|x|$). In the implementation below,
the polynomial $P(x)$ is constructed with the technical constraint $P(0)=0$,
meaning $P(x)$ has a factor of $x^2$ - the arguments for this reduction to a more tractable class of functions are given e.g. in Section 3 of Gonçalves et al. (2017). The largest positive root of $P(x)$ is
then the largest positive root of $P(x)/x^2$. The upper bound on $C_4$ is given
by this largest root squared, divided by $2\pi$.

AlphaEvolve found the coefficients ($c_0$, $c_1$, $c_2$) defining a polynomial
$P(x) = c_0 H_0(x) + c_1 H_4(x) + c_2 H_8(x)$ that provides an improved upper
bound for $C_4$.

In [ ]:
#@title Data & utility functions

coefficients_sota = [
    -113 / 100,
    1 / 25,
    1 / 3240,
]

coefficients_alphaevolve = [
    0.3292519302257546,
    -0.01158510802599293,
    -8.921606035407065e-05
]

In [ ]:
#@title Verifier


def verify_hermite_combination(poly_expr: sympy.Expr):
  """Validates that a polynomial meets required Hermite combination constraints.

  This function performs two checks:
  1. Roots: Verifies the polynomial passes through the origin (g(0) = 0).
  2. Stability: Verifies the polynomial diverges to positive infinity.

  Args:
    poly_expr: The SymPy expression (polynomial) to be validated.

  Raises:
    AssertionError: If the value at zero is non-zero or the limit at
                    infinity is not positive.
  """
  x = sympy.symbols('x')

  # Requirement 1: The polynomial must be 'anchored' at the origin.
  value_at_0 = poly_expr.subs(x, 0)
  assert value_at_0 == 0, f"Constraint Violation: g(0) should be 0, but got {value_at_0}."

  # Requirement 2: Ensure the function doesn't flip 'upside down' at large scales.
  limit_at_infty = sympy.limit(poly_expr, x, sympy.oo)
  assert limit_at_infty > 0, f"Constraint Violation: Limit at ∞ must be positive, but got {limit_at_infty}."


def compute_hermite_combination(coeffs: list[float]) -> sympy.Expr:
  """Constructs a linear combination of Hermite polynomials given coefficients.

  The resulting polynomial:
  1. Uses degrees 0, 4, 8, ..., 4m.
  2. Has a root at x=0 by calculating a specific final coefficient.
  3. Is normalized to be positive as x approaches infinity.

  Args:
    coeffs: A list of real coefficients c_i.

  Returns:
    A sympy expression for the linear combination of the Hermite polynomials.
  """
  m = len(coeffs)
  x = sympy.symbols('x')

  # Map degrees to 0, 4, 8, 12, etc.
  degrees = np.arange(0, 4 * m + 4, 4)
  rational_coeffs = [sympy.Rational(c) for c in coeffs]

  # Generate Hermite polynomials H_0, H_4, H_8, etc.
  hermite_polys = [
      sympy.polys.orthopolys.hermite_poly(n=d, x=x, polys=False)
      for d in degrees
  ]

  # Sum initial terms: partial_result = c0*H0 + c4*H4 + etc.
  partial_result = sympy.Add(
      *(c * h for c, h in zip(rational_coeffs, hermite_polys))
  )

  # Solve for the final coefficient so that final_result(0) = 0.
  # Equation: partial_result(0) + last_coeff * last_hermite(0) = 0.
  h_last_at_zero = hermite_polys[-1].subs(x, 0)
  partial_at_zero = partial_result.subs(x, 0)

  last_coeff = sympy.Rational(-partial_at_zero / h_last_at_zero)
  rational_coeffs.append(last_coeff)

  # Reconstruct the full polynomial.
  full_poly = sum(c * h for c, h in zip(rational_coeffs, hermite_polys))

  # Ensure leading behavior is positive (positive at infinity).
  if sympy.limit(full_poly, x, sympy.oo) < 0:
    full_poly = -full_poly

  return full_poly


# Precision constants for sign-change detection
PRECISION = 200
EPSILON = sympy.Rational(1, 10**198)  # Equivalent to 1e-198


def compute_upper_bound(coeffs: list[float]) -> float:
  """Evaluates score of a coefficient set based on roots of the Hermite poly.

  The score identifies the largest real x-value where the polynomial crosses
  the x-axis (sign change). The score is returned as -x^2 / 2π.

  Args:
    coeffs: A list of real coefficients c_i. Must contain 3 floats. The maximum
      absolute value of coefficients must be between 1e-14 and 1000.

  Returns:
    The final score of the coefficients.
  """
  if (
      not isinstance(coeffs, list)
      or len(coeffs) != 3
      or not all(isinstance(x, (int, float, np.floating)) for x in coeffs)
      or max([abs(x) for x in coeffs]) > 1000
      or max([abs(x) for x in coeffs]) < 1e-14
  ):
    return float('-inf')

  x = sympy.symbols('x')
  poly_expr = compute_hermite_combination(coeffs)

  # Divide by x^2 to remove the known trivial root at the origin.
  reduced_poly = sympy.exquo(poly_expr, x**2)

  # Find all real roots of the remaining polynomial.
  real_roots = sympy.real_roots(reduced_poly, x)

  largest_crossing_root = 0

  for root in real_roots:
    # High-precision rational approximation.
    approx_root = root.eval_rational(n=PRECISION)

    # Check for sign change around the root (verifies it's not just a tangent).
    val_plus = reduced_poly.subs(x, approx_root + EPSILON)
    val_minus = reduced_poly.subs(x, approx_root - EPSILON)

    crosses_axis = (val_plus * val_minus) < 0

    if crosses_axis:
      largest_crossing_root = max(largest_crossing_root, approx_root)

  # Return the normalized upper bound.
  return float(largest_crossing_root**2) / (2 * np.pi)


sota_exp = compute_hermite_combination(coefficients_sota)
alphaevolve_exp = compute_hermite_combination(coefficients_alphaevolve)

verify_hermite_combination(sota_exp)
verify_hermite_combination(alphaevolve_exp)

print(
    'The construction of Gonçalves et al. (2017) is correct and gives the '
    'upper bound: C4 <=',
    compute_upper_bound(coefficients_sota)
)
print(
    'AlphaEvolve construction is correct and gives the upper bound: C4 <=',
    compute_upper_bound(coefficients_alphaevolve)
)

In [ ]:
# @title Plotting

x = sympy.symbols("x")
sota_fn = sympy.lambdify(x, sota_exp, modules=["numpy"])
sota_total_fn = lambda x: np.exp(-np.pi * x**2) * sota_fn(
    np.sqrt(2 * np.pi) * x
)

alphaevolve_fn = sympy.lambdify(x, alphaevolve_exp, modules=["numpy"])
alphaevolve_total_fn = lambda x: np.exp(-np.pi * x**2) * alphaevolve_fn(
    np.sqrt(2 * np.pi) * x
)

x_vals = np.linspace(-5, 5, 1000)

plt.figure(figsize=(8, 5))

plt.plot(
    x_vals, sota_total_fn(x_vals), label="Gonçalves et al. (2017)", color="red"
)

plt.plot(
    x_vals,
    alphaevolve_total_fn(x_vals),
    label="AlphaEvolve",
    color="green",
)

plt.axhline(0, color="black", linestyle="--", linewidth=0.5)  # x-axis
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title(
    "Gonçalves et al. (2017) and AlphaEvolve's discovered function for the"
    " uncertainty inequality"
)
plt.legend()
plt.grid(True)
plt.show()